# Module 16 - Feature Engineering Mini Challenge

## 1. Understand the Dataset

The Hotel Bookings dataset contains information about hotel reservations, guests, stay duration, booking details, pricing, arrival dates, and reservation outcomes.

The objective of this mini challenge is to independently identify useful information in the dataset and create meaningful features without relying on predefined feature-engineering instructions.

Before creating new features, we need to understand:

- Dataset size
- Available columns
- Data types
- Missing values
- Numerical and categorical features
- Overall structure of the dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\HP\Sprint_6_Feature_Engineering_-_Feature_Selection\data\hotel_bookings.csv"
)

print("Shape:", df.shape)

df.head()

Shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [3]:
print("Missing Values:")
print(df.isnull().sum())

print("\nNumerical Features:")
print(df.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical Features:")
print(df.select_dtypes(include="object").columns.tolist())

Missing Values:
hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340


C:\Users\HP\AppData\Local\Temp\ipykernel_6204\1451191637.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include="object").columns.tolist())


The Hotel Bookings dataset contains booking, guest, stay, pricing, arrival-date, and reservation information.

The dataset contains both numerical and categorical features.

Before feature engineering, we need to understand the available information and determine which business question and target variable are appropriate for this dataset.

## Step 2: Identify the Target

Business question: **Will a hotel booking be canceled?**

This is a **booking cancellation prediction** problem. The target is `is_canceled`.

Target meaning:
- `0` → Booking was not canceled
- `1` → Booking was canceled

The target will be kept separate from the input features during feature engineering and modeling.

We must also make sure that no post-outcome information is used to create features for predicting cancellation.

In [4]:
target = "is_canceled"

print("Target column:", target)
print("\nTarget distribution:")
print(df[target].value_counts())

print("\nTarget percentage:")
print(df[target].value_counts(normalize=True) * 100)

Target column: is_canceled

Target distribution:
is_canceled
0    75166
1    44224
Name: count, dtype: int64

Target percentage:
is_canceled
0    62.958372
1    37.041628
Name: proportion, dtype: float64


### Observation

The target variable for this mini challenge is `is_canceled`.

The objective is to use booking-time information to identify patterns associated with whether a reservation is canceled.

Features such as `reservation_status` and `reservation_status_date` will not be used as predictive inputs because they contain information related to the reservation outcome.

## Step 3: Identify Existing (Raw) Features

Before creating new features, identify the raw columns already available in the Hotel Bookings dataset.

| Column | Type | Usefulness |
|---|---|---|
| `hotel` | Categorical | Useful for identifying hotel type |
| `lead_time` | Numerical | Useful for booking behavior |
| `arrival_date_year` | Numerical | Useful for time-based patterns |
| `arrival_date_month` | Categorical | Useful for seasonality |
| `arrival_date_week_number` | Numerical | Useful for time-based patterns |
| `arrival_date_day_of_month` | Numerical | Useful for arrival timing |
| `stays_in_weekend_nights` | Numerical | Useful for stay duration |
| `stays_in_week_nights` | Numerical | Useful for stay duration |
| `adults` | Numerical | Useful for guest composition |
| `children` | Numerical | Useful for guest composition |
| `babies` | Numerical | Useful for guest composition |
| `meal` | Categorical | Useful for booking characteristics |
| `market_segment` | Categorical | Useful for customer/booking segment |
| `distribution_channel` | Categorical | Useful for booking source |
| `adr` | Numerical | Useful for pricing |
| `total_of_special_requests` | Numerical | Useful for guest behavior |
| `customer_type` | Categorical | Useful for customer type |
| `is_canceled` | Target | Prediction target |

Some columns describe raw booking information but do not directly capture relationships such as total stay duration, total guests, booking value, or guest composition.

This is why feature engineering is required.

In [5]:
print("Existing Features:")
print(df.columns.tolist())

print("\nNumber of Features:", len(df.columns))

Existing Features:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']

Number of Features: 32


### Observation

The dataset contains numerical and categorical booking features along with the target `is_canceled`.

The raw features provide useful information, but several meaningful business signals can be derived by combining or transforming them.

These gaps will be addressed in the next step.

## Step 4: Identify Potentially Useful New Features

Based on the business question — predicting whether a hotel booking will be canceled — we need features that capture booking behavior, stay characteristics, guest information, pricing, and timing.

Potentially useful new features include:

- Total stay duration
- Total number of guests
- Weekend stay indicator
- Long-stay indicator
- Booking lead-time category
- Guests per night
- ADR per guest
- Estimated stay revenue
- Family booking indicator
- Seasonal booking
- Special request rate
- Guest composition
- Arrival day information
- Feature interactions

These features are derived from the existing raw columns and provide additional business meaning that is not directly represented by individual columns.

In [6]:
potential_features = [
    "Total_Stay_Nights",
    "Total_Guests",
    "Is_Weekend_Stay",
    "Is_Long_Stay",
    "Booking_Lead_Category",
    "Guests_Per_Night",
    "ADR_Per_Guest",
    "Estimated_Stay_Revenue",
    "Family_Booking",
    "Seasonal_Booking",
    "Special_Request_Rate",
    "Guest_Composition",
    "Arrival_Date",
    "Is_Weekend_Arrival",
    "LeadTime_ADR_Interaction",
    "Stay_ADR_Interaction",
    "Guests_Stay_Interaction"
]

print("Potential New Features:")
for feature in potential_features:
    print("-", feature)

Potential New Features:
- Total_Stay_Nights
- Total_Guests
- Is_Weekend_Stay
- Is_Long_Stay
- Booking_Lead_Category
- Guests_Per_Night
- ADR_Per_Guest
- Estimated_Stay_Revenue
- Family_Booking
- Seasonal_Booking
- Special_Request_Rate
- Guest_Composition
- Arrival_Date
- Is_Weekend_Arrival
- LeadTime_ADR_Interaction
- Stay_ADR_Interaction
- Guests_Stay_Interaction


### Observation

The existing booking information provides several opportunities for meaningful feature engineering.

The selected features cover different aspects of hotel bookings:

- Stay behavior
- Guest characteristics
- Pricing
- Booking timing
- Seasonality
- Guest requests
- Feature interactions

These candidate features will be created and evaluated in the next step.

## Step 5: Create at Least 10 Meaningful Features

The following features are created from the raw Hotel Bookings data.

All features are based on booking information that can be available before the final reservation outcome.

We will create more than the required 10 features to capture different aspects of hotel booking behavior.

In [7]:
df["children"] = df["children"].fillna(0)

# 1. Total stay duration
df["Total_Stay_Nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

# 2. Total guests
df["Total_Guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

# 3. Weekend stay indicator
df["Is_Weekend_Stay"] = (
    df["stays_in_weekend_nights"] > 0
).astype(int)

# 4. Long stay indicator
df["Is_Long_Stay"] = (
    df["Total_Stay_Nights"] > 7
).astype(int)

# 5. Booking lead-time category
df["Booking_Lead_Category"] = pd.cut(
    df["lead_time"],
    bins=[-1, 30, 90, np.inf],
    labels=["Short Lead", "Medium Lead", "Long Lead"]
)

# 6. Guests per night
df["Guests_Per_Night"] = np.where(
    df["Total_Stay_Nights"] > 0,
    df["Total_Guests"] / df["Total_Stay_Nights"],
    0
)

# 7. ADR per guest
df["ADR_Per_Guest"] = np.where(
    df["Total_Guests"] > 0,
    df["adr"] / df["Total_Guests"],
    0
)

# 8. Estimated stay revenue
df["Estimated_Stay_Revenue"] = (
    df["adr"] * df["Total_Stay_Nights"]
)

# 9. Family booking
df["Family_Booking"] = (
    (df["children"] > 0) |
    (df["babies"] > 0)
).astype(int)

# 10. Seasonal booking
season_map = {
    "December": "Winter",
    "January": "Winter",
    "February": "Winter",
    "March": "Spring",
    "April": "Spring",
    "May": "Spring",
    "June": "Summer",
    "July": "Summer",
    "August": "Summer",
    "September": "Autumn",
    "October": "Autumn",
    "November": "Autumn"
}

df["Seasonal_Booking"] = (
    df["arrival_date_month"].map(season_map)
)

# 11. Special request rate
df["Special_Request_Rate"] = np.where(
    df["Total_Stay_Nights"] > 0,
    df["total_of_special_requests"] / df["Total_Stay_Nights"],
    0
)

# 12. Guest composition
def get_guest_composition(row):
    if row["children"] > 0 or row["babies"] > 0:
        return "Family"
    elif row["adults"] == 1:
        return "Solo"
    else:
        return "Adults Only"

df["Guest_Composition"] = df.apply(
    get_guest_composition,
    axis=1
)

# 13. Arrival date
month_map = {
    "January": 1, "February": 2, "March": 3,
    "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9,
    "October": 10, "November": 11, "December": 12
}

df["Arrival_Month_Number"] = (
    df["arrival_date_month"].map(month_map)
)

df["Arrival_Date"] = pd.to_datetime(
    dict(
        year=df["arrival_date_year"],
        month=df["Arrival_Month_Number"],
        day=df["arrival_date_day_of_month"]
    ),
    errors="coerce"
)

# 14. Weekend arrival
df["Arrival_DayOfWeek"] = (
    df["Arrival_Date"].dt.dayofweek
)

df["Is_Weekend_Arrival"] = (
    df["Arrival_DayOfWeek"] >= 5
).astype(int)

# 15. Lead time × ADR
df["LeadTime_ADR_Interaction"] = (
    df["lead_time"] * df["adr"]
)

# 16. Stay × ADR
df["Stay_ADR_Interaction"] = (
    df["Total_Stay_Nights"] * df["adr"]
)

# 17. Guests × Stay
df["Guests_Stay_Interaction"] = (
    df["Total_Guests"] * df["Total_Stay_Nights"]
)

new_features = [
    "Total_Stay_Nights",
    "Total_Guests",
    "Is_Weekend_Stay",
    "Is_Long_Stay",
    "Booking_Lead_Category",
    "Guests_Per_Night",
    "ADR_Per_Guest",
    "Estimated_Stay_Revenue",
    "Family_Booking",
    "Seasonal_Booking",
    "Special_Request_Rate",
    "Guest_Composition",
    "Arrival_Date",
    "Is_Weekend_Arrival",
    "LeadTime_ADR_Interaction",
    "Stay_ADR_Interaction",
    "Guests_Stay_Interaction"
]

print("Number of new features:", len(new_features))
print("\nNew Features:")
print(new_features)

Number of new features: 17

New Features:
['Total_Stay_Nights', 'Total_Guests', 'Is_Weekend_Stay', 'Is_Long_Stay', 'Booking_Lead_Category', 'Guests_Per_Night', 'ADR_Per_Guest', 'Estimated_Stay_Revenue', 'Family_Booking', 'Seasonal_Booking', 'Special_Request_Rate', 'Guest_Composition', 'Arrival_Date', 'Is_Weekend_Arrival', 'LeadTime_ADR_Interaction', 'Stay_ADR_Interaction', 'Guests_Stay_Interaction']


### Observation

17 meaningful features have been created from the Hotel Bookings dataset.

The features capture:

- Stay duration
- Guest information
- Booking timing
- Pricing
- Revenue
- Guest composition
- Seasonality
- Special requests
- Arrival timing
- Feature interactions

This exceeds the minimum requirement of 10 meaningful features for the mini challenge.

## Step 6: Explain Every Feature

Each engineered feature is documented with its business meaning and relevance to the hotel booking problem.

1. **Total_Stay_Nights** – Total number of weekend and weekday nights. Represents the planned duration of the stay.

2. **Total_Guests** – Total number of adults, children, and babies. Represents the size of the booking.

3. **Is_Weekend_Stay** – Indicates whether the booking contains one or more weekend nights.

4. **Is_Long_Stay** – Indicates whether the planned stay is longer than 7 nights.

5. **Booking_Lead_Category** – Groups bookings into Short Lead, Medium Lead, and Long Lead based on `lead_time`.

6. **Guests_Per_Night** – Total guests divided by total stay nights. Represents guest count relative to stay duration.

7. **ADR_Per_Guest** – Average Daily Rate divided by total guests. Represents pricing relative to guest count.

8. **Estimated_Stay_Revenue** – ADR multiplied by total stay nights. Represents the estimated value of the planned stay.

9. **Family_Booking** – Indicates whether children or babies are included in the booking.

10. **Seasonal_Booking** – Converts the arrival month into a seasonal category.

11. **Special_Request_Rate** – Special requests divided by total stay nights. Represents request frequency relative to stay duration.

12. **Guest_Composition** – Categorizes bookings as Family, Solo, or Adults Only.

13. **Arrival_Date** – Combines the arrival year, month, and day into a complete date.

14. **Is_Weekend_Arrival** – Indicates whether the arrival date falls on Saturday or Sunday.

15. **LeadTime_ADR_Interaction** – Combines booking lead time and ADR to capture their interaction.

16. **Stay_ADR_Interaction** – Combines total stay duration and ADR to represent their joint effect.

17. **Guests_Stay_Interaction** – Combines total guests and stay duration to capture the relationship between booking size and duration.

These features were created from existing booking information and are evaluated based on their business usefulness, ML relevance, and potential leakage risk.

## Step 7: Check for Leakage

Feature leakage occurs when information that would not be available at prediction time is used to create a feature.

For this Hotel Bookings challenge, the target is `is_canceled`.

The following checks are performed:

- Do not use `reservation_status` or `reservation_status_date` as model features because they describe the reservation outcome.
- Do not use the target `is_canceled` while creating features.
- Ensure engineered features are created only from booking-time information.
- Avoid using any information that becomes available only after the cancellation outcome.

The engineered features created in this challenge are based on booking, guest, stay, pricing, and arrival information.

In [8]:
leakage_features = [
    "reservation_status",
    "reservation_status_date",
    "is_canceled"
]

print("Potential leakage features:")

for feature in leakage_features:
    if feature in df.columns:
        print("-", feature)

Potential leakage features:
- reservation_status
- reservation_status_date
- is_canceled


### Leakage Check Result

`reservation_status` and `reservation_status_date` are identified as post-outcome features and must not be used as model inputs.

The target `is_canceled` is kept separate from feature creation.

Therefore, the engineered features are designed using information that can be available before the final cancellation outcome.

## Step 8: Remove Irrelevant Features

Before feature selection, remove features that should not be used as model inputs.

The following features are removed:

- `reservation_status` – post-outcome information and causes target leakage.
- `reservation_status_date` – contains post-outcome information.
- `is_canceled` – target variable, not an input feature.
- `Arrival_Date` – raw datetime feature; useful date components have already been extracted.
- `Arrival_Month_Number` – intermediate column used only to construct `Arrival_Date`.

The goal is to keep useful engineered features while avoiding leakage, redundant information, and intermediate processing columns.

In [9]:
irrelevant_features = [
    "reservation_status",
    "reservation_status_date",
    "is_canceled",
    "Arrival_Date",
    "Arrival_Month_Number"
]

model_df = df.drop(
    columns=irrelevant_features,
    errors="ignore"
)

print("Remaining columns:", len(model_df.columns))
print(model_df.columns.tolist())

Remaining columns: 46
['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'Total_Stay_Nights', 'Total_Guests', 'Is_Weekend_Stay', 'Is_Long_Stay', 'Booking_Lead_Category', 'Guests_Per_Night', 'ADR_Per_Guest', 'Estimated_Stay_Revenue', 'Family_Booking', 'Seasonal_Booking', 'Special_Request_Rate', 'Guest_Composition', 'Arrival_DayOfWeek', 'Is_Weekend_Arrival', 'LeadTime_ADR_Interaction', 'Stay_ADR_Interaction', 'Guests_Stay_Interaction']


### Observation

The unnecessary and leakage-prone columns have been removed from the working feature dataset.

The target `is_canceled` is kept separately for the feature-selection stage.

The remaining dataset contains original and engineered features that can be evaluated for their usefulness in predicting booking cancellation.

## Step 9: Perform Feature Selection

After removing irrelevant and leakage-prone features, we need to identify which remaining features provide useful predictive information.

Two simple approaches will be used:

1. Correlation with the target — checks the linear relationship between numerical features and `is_canceled`.
2. Random Forest Feature Importance — captures non-linear relationships and ranks features based on their contribution to the model.

In [10]:
from sklearn.ensemble import RandomForestClassifier

# Keep numerical features for selection
numeric_features = model_df.select_dtypes(
    include=np.number
).columns.tolist()

X = model_df[numeric_features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

y = df["is_canceled"]

# Correlation with target
correlations = (
    X.assign(is_canceled=y)
    .corr()["is_canceled"]
    .drop("is_canceled")
    .sort_values(ascending=False)
)

print("Correlation with Target:")
print(correlations)

Correlation with Target:
lead_time                         0.293123
LeadTime_ADR_Interaction          0.258424
previous_cancellations            0.110133
adults                            0.060017
days_in_waiting_list              0.054186
adr                               0.047557
Estimated_Stay_Revenue            0.046562
Stay_ADR_Interaction              0.046562
Total_Guests                      0.046522
Guests_Stay_Interaction           0.031888
stays_in_week_nights              0.024765
Arrival_DayOfWeek                 0.022146
ADR_Per_Guest                     0.020830
Total_Stay_Nights                 0.017779
arrival_date_year                 0.016660
arrival_date_week_number          0.008148
children                          0.005036
stays_in_weekend_nights          -0.001791
Is_Long_Stay                     -0.005772
arrival_date_day_of_month        -0.006130
Is_Weekend_Stay                  -0.008399
Is_Weekend_Arrival               -0.009103
Family_Booking               

## Step 10: Compare the Dataset Before and After Feature Engineering

The final step is to compare the dataset before and after feature engineering.

This comparison helps us understand how feature engineering changed the dataset and whether additional business information was captured.

We will compare:

- Number of rows
- Number of columns
- Original features
- Engineered features
- Target availability
- ML readiness

In [12]:
original_features = df.columns.tolist()

engineered_features = [
    feature for feature in new_features
    if feature in df.columns
]

comparison = pd.DataFrame({
    "Before Feature Engineering": [
        df.shape[0],
        len(original_features),
        "Raw booking features",
        "Yes",
        "Requires feature preparation"
    ],
    "After Feature Engineering": [
        df.shape[0],
        len(df.columns),
        "Raw + engineered features",
        "Yes",
        "Ready for selection and ML preprocessing"
    ]
}, index=[
    "Rows",
    "Columns",
    "Feature Type",
    "Target Available",
    "ML Readiness"
])

comparison

,Before Feature Engineering,After Feature Engineering
Rows,119390,119390
Columns,51,51
Feature Type,Raw booking features,Raw + engineered features
Target Available,Yes,Yes
ML Readiness,Requires feature preparation,Ready for selection and ML preprocessing


In [13]:
print("Engineered Features Created:", len(engineered_features))

print("\nEngineered Features:")
print(engineered_features)

Engineered Features Created: 17

Engineered Features:
['Total_Stay_Nights', 'Total_Guests', 'Is_Weekend_Stay', 'Is_Long_Stay', 'Booking_Lead_Category', 'Guests_Per_Night', 'ADR_Per_Guest', 'Estimated_Stay_Revenue', 'Family_Booking', 'Seasonal_Booking', 'Special_Request_Rate', 'Guest_Composition', 'Arrival_Date', 'Is_Weekend_Arrival', 'LeadTime_ADR_Interaction', 'Stay_ADR_Interaction', 'Guests_Stay_Interaction']


### Observation

The dataset now contains the original booking information along with 17 engineered features.

Feature engineering added information related to:

- Stay duration
- Guest composition
- Booking timing
- Pricing
- Estimated booking value
- Seasonality
- Special requests
- Arrival timing
- Feature interactions

The engineered dataset provides richer information than the original raw features and can now proceed to the final ML preprocessing and modeling workflow.